In [ ]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

In [ ]:
import pandas as pd

def clean_ro_series(s):
    return (
        s.fillna("").astype(str)
        .str.replace("ţ", "ț")
        .str.replace("ş", "ș")
        .str.replace("Ţ", "Ț")
        .str.replace("Ş", "Ș")
    )

IN_DIR = "/kaggle/input/datasets/ababeiraul/msiddataset"

train_df = pd.read_csv(f"{IN_DIR}/train_data.csv")
test_df = pd.read_csv(f"{IN_DIR}/test_data.csv")

X_train_txt = clean_ro_series(train_df["sample"]).reset_index(drop = True)
X_test_txt = clean_ro_series(test_df["sample"]).reset_index(drop = True)
y2 = (train_df["category"] - 1).values


In [ ]:
MODEL_NAME = "dumitrescustefan/bert-base-romanian-cased-v1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)

In [ ]:
emb_matrix = bert_model.get_input_embeddings().weight.detach().numpy()
special_ids = set(tokenizer.all_special_ids)

In [ ]:
def get_static_embeddings(texts):
    vectors = np.zeros((len(texts), emb_matrix.shape[1]), dtype=np.float32)
    for i, text in enumerate(texts):
        # remove our special token
        clean_t = text.replace("$NE$", " ").replace("NE", " ")
        ids = [tid for tid in tokenizer.encode(clean_t, truncation=True, max_length=256) if tid not in special_ids]
        if ids:
            vectors[i] = emb_matrix[ids].mean(axis=0)
    return vectors

In [ ]:
X_train_emb = get_static_embeddings(X_train_txt)
X_test_emb = get_static_embeddings(X_test_txt)
print("Dimensiune vectori RoBERT statici:", X_train_emb.shape)

In [ ]:
strkf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_emb = np.zeros((len(train_df), 6))
test_emb = np.zeros((len(test_df), 6))

oof_tfidf = np.zeros((len(train_df), 6))
test_tfidf = np.zeros((len(test_df), 6))

vec = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 2), max_features=60000, sublinear_tf=True, lowercase=False)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 6), max_features=80000, sublinear_tf=True, lowercase=False)),
])

In [ ]:
for tr_idx, val_idx in strkf.split(X_train_txt, y2):
    # LogisticRegression on static vectors RoBERT
    scaler = StandardScaler()
    E_tr = scaler.fit_transform(X_train_emb[tr_idx])
    E_val = scaler.transform(X_train_emb[val_idx])
    E_te = scaler.transform(X_test_emb)

    clf_emb = LogisticRegression(C=0.05, max_iter=1000).fit(E_tr, y2[tr_idx])
    oof_emb[val_idx] = clf_emb.predict_proba(E_val)
    test_emb += clf_emb.predict_proba(E_te) / 5.0

    # naive solution we developed previously
    T_tr = vec.fit_transform(X_train_txt.iloc[tr_idx])
    T_val = vec.transform(X_train_txt.iloc[val_idx])
    T_te = vec.transform(X_test_txt)

    clf_tfidf = LogisticRegression(C=1.5, max_iter=1000).fit(T_tr, y2[tr_idx])
    oof_tfidf[val_idx] = clf_tfidf.predict_proba(T_val)
    test_tfidf += clf_tfidf.predict_proba(T_te) / 5.0

In [ ]:
print("CV F1 doar RoBERT static :", round(f1_score(y2, np.argmax(oof_emb, axis=1), average="weighted"), 4))
print("CV F1 doar TF-IDF        :", round(f1_score(y2, np.argmax(oof_tfidf, axis=1), average="weighted"), 4))

In [ ]:
best_w, best_f1 = 0.0, 0.0
for w in np.linspace(0, 1, 21):
    blend = w * oof_emb + (1 - w) * oof_tfidf
    score = f1_score(y2, np.argmax(blend, axis=1), average="weighted")
    if score > best_f1:
        best_f1, best_w = score, w

print(f"CV F1 COMBINAT           : {best_f1:.4f} (RoBERT_static={best_w:.2f}, TF-IDF={1 - best_w:.2f})")

In [ ]:
final_probs_t2 = best_w * test_emb + (1 - best_w) * test_tfidf
task2_preds = np.argmax(final_probs_t2, axis=1) + 1

In [ ]:
sub2 = pd.DataFrame({"subtaskID": 2, "datapointID": test_df["datapointID"], "answer": task2_preds})

sub2.to_csv("submission_task1_2.csv", index=False)